# Data Validation

This notebook validates the data loaded into Snowflake for the BlueTide SaaS churn and retention project. This notebook ensures all tables are complete, correctly typed, and consistent before proceeding with analysis. 

## Business Context

- Database `BLUETIDE` and schema `ANALYTICS` have been created in Snowflake.  
- All five tables (`companies`, `users`, `subscriptions`, `product_usage`, `support_tickets`) were loaded from CSV exports generated in notebook 1.   
- This notebook focuses on validating row counts, key columns, foreign key integrity, and overall distributions.

## Validation Plan

1. **Row counts** — make sure each table has the expected number of records.  
2. **Null checks** — confirm key columns (IDs, company_id) are not missing.  
3. **Foreign key integrity** — ensure users, subscriptions, usage, and tickets all point to valid companies.  
4. **Distribution sanity checks** — check plan types, statuses, and activity metrics for reasonableness.

### Row Counts


In [ ]:
-- get the number of rows in each table

SELECT 'companies' AS table_name, COUNT(*) AS row_count FROM companies
UNION ALL
SELECT 'users', COUNT(*) FROM users
UNION ALL
SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALLD
SELECT 'product_usage', COUNT(*) FROM product_usage
UNION ALL
SELECT 'support_tickets', COUNT(*) FROM support_tickets;

**Expected results**
| table_name      | row_count |
| --------------- | --------- |
| companies       | 500       |
| users           | 13562     |
| subscriptions   | 500       |
| product_usage   | 45000     |
| support_tickets | 2502      |


All tables loaded successfully and match the expected record counts from the data generation step.

### Null Checks for Primary Keys

In [ ]:
-- find missing company ids
SELECT COUNT(*) AS missing_company_ids
FROM companies
WHERE company_id IS NULL;

There are no missing company ids. 

In [ ]:
-- find missing user ids
SELECT COUNT(*) AS missing_user_ids
FROM users
WHERE user_id IS NULL;

There are no missing user IDs. 

In [ ]:
-- find missing subscription ids
SELECT COUNT(*) AS missing_subscription_ids
FROM subscriptions
WHERE subscription_id IS NULL;

There are no missing subscription IDs. 

In [ ]:
-- find missing usage ids
SELECT COUNT(*) AS missing_usage_ids
FROM product_usage
WHERE usage_id IS NULL;

There are no missing usage IDs. 

In [ ]:
-- find missing support ticket ids
SELECT COUNT(*) AS missing_ticket_ids
FROM support_tickets
WHERE ticket_id IS NULL;

All primary keys contain valid values and no null records were detected.

### Foreign Key Integrity


In [ ]:
-- do users belong to a valid company?
SELECT COUNT(*) AS orphan_users
FROM users u
LEFT JOIN companies c
ON u.company_id = c.company_id
WHERE c.company_id IS NULL;

There are no orphaned users. 

In [ ]:
-- do subscriptions belong to a valid company?
SELECT COUNT(*) AS orphan_subscriptions
FROM subscriptions s
LEFT JOIN companies c
ON s.company_id = c.company_id
WHERE c.company_id IS NULL;

There are no orphaned subscriptions.

In [ ]:
-- does product usage belong to a valid company?
SELECT COUNT(*) AS orphan_usage_records
FROM product_usage p
LEFT JOIN companies c
ON p.company_id = c.company_id
WHERE c.company_id IS NULL;

There are no orphaned usage records.

In [ ]:
-- do support tickets belong to a valid company?
SELECT COUNT(*) AS orphan_tickets
FROM support_tickets t
LEFT JOIN companies c
ON t.company_id = c.company_id
WHERE c.company_id IS NULL;

All tables correctly reference existing companies. No orphan records were found.

### Distribution Checks

In [ ]:
-- company status
SELECT status, COUNT(*) AS company_count
FROM companies
GROUP BY status;

There are 100 churned and 400 active customers. 

In [ ]:
-- subscription plans
SELECT plan_name, COUNT(*) AS plan_count
FROM subscriptions
GROUP BY plan_name;

The output matches our simulation. 

|Plan Name|Plan Count|
|---|---|
|Pro|117|
|Free|123|
|Enterprise|121|
|Team|1396|

In [ ]:
-- payment status 
SELECT payment_status, COUNT(*) AS payment_count
FROM subscriptions
GROUP BY payment_status;

Plan distribution, company status, and payment behavior appear consistent with expectations for a SaaS customer base.

|Payment Status|Payment Count|
|---|---|
|Late|257|
|Paid|193|
|Failed|50|

### Product Usage Summary

In [ ]:
-- how much usage does each company get out of their plan?
SELECT
    company_id,
    SUM(active_users) AS total_active_users,
    SUM(commits) AS total_commits,
    SUM(pipelines_run) AS total_pipelines,
    SUM(deployments) AS total_deployments
FROM product_usage
GROUP BY company_id
ORDER BY total_commits DESC
LIMIT 10;

In [ ]:
-- how many commits do companies usually make
SELECT
    AVG(commits) AS avg_commits,
    MAX(commits) AS max_commits,
    MIN(commits) AS min_commits
FROM product_usage;

Product activity metrics appear realistic and provide a strong signal for future churn analysis.

## Final Validation Summary
The BlueTide dataset successfully passed all validation checks:
- All tables loaded correctly
- Primary keys contain no null values
- Foreign key relationships are intact
- Customer distributions appear realistic
- Product usage metrics are populated and analyzable

The dataset is now ready for exploratory analysis and churn modeling in the next notebook.